In [ ]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [ ]:
import os

response = openai_client.chat.completions.create(
    model=os.environ["OPENAI_MODEL"],
    messages=[{"role": "user", "content": "Reply with exactly one word: hello"}] # thinking models can burn the budget before content appears
)

choice = response.choices[0]
msg = choice.message
text = msg.content or getattr(msg, "reasoning", None)

print("Routed model:", response.model)
print("Finish reason:", choice.finish_reason)
print("Response:", text)

In [ ]:
def llm(instructions, prompt=None, model=None):
    if model is None:
        model = os.environ["OPENAI_MODEL"]

    if prompt is None:
        message_history = [{"role": "user", "content": instructions}]
    else:
        message_history = [
            {"role": "developer", "content": instructions},
            {"role": "user", "content": prompt},
        ]

    response = openai_client.responses.create(
        model=model,
        input=message_history,
        max_output_tokens=1024,
    )
    return response.output_text

In [ ]:
llm("Hey, what's up?")

In [ ]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

In [ ]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [ ]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
answer = llm(prompt)
print(answer)

In [ ]:
print (prompt)

In [ ]:
import requests


docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

print (courses_raw)

In [ ]:
from pprint import pprint

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

print(len(documents))
pprint (documents[0])

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

In [ ]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

[doc["question"] for doc in results]

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )


In [ ]:
search_results = search(question)
search_results

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [ ]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""
USER_PROMPT_TEMPLATE.format(question="questin", context="context")

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [ ]:
contex = build_context(search_results)
print(context)

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [ ]:
prompt = build_prompt(question, search_results)

print(prompt)

In [ ]:
response = openai_client.responses.create(
    model=os.environ["OPENAI_MODEL"],
    input=prompt
)

response.output_text

In [ ]:
print(response.model_dump_json(indent=2))

In [ ]:
response.output[0].content[0].text

In [ ]:
response.usage

In [ ]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

In [ ]:
message_history = [
    {'role': 'developer', 'content': INSTRUCTIONS},
    {'role': 'user', 'content': prompt}
]

response = openai_client.responses.create(
    model=os.environ["OPENAI_MODEL"],
    input=message_history,
    max_output_tokens=1024,
)

In [ ]:
def rag(query, model=None):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [ ]:
answer = rag(question)
print(answer)